# SHAP Explainability — Mitigated Models

This notebook evaluates how bias mitigation strategies affect feature importance using SHAP.

It compares:
- Baseline model behaviour (see previous notebook)
- Reweighing (pre-processing)
- Exponentiated Gradient (in-processing)

The goal is to assess whether fairness interventions alter model reasoning or primarily affect decision boundaries.

In [2]:
import sys
from pathlib import Path
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


# Add project root
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.data.loader import load_hr_analytics
from src.models.baseline import get_logistic_regression_pipeline

output_dir = Path("../runs/shap_mitigated")
output_dir.mkdir(parents=True, exist_ok=True)



## Reweighing (Pre-processing)

This section evaluates whether reweighing alters feature importance by adjusting sample weights during training.

In [3]:
ds = load_hr_analytics("../data/raw/HR_Analytics.csv")

from src.mitigation.reweighing import (
    to_aif360_dataset,
    apply_reweighing,
    extract_weights,
)

# Create AIF360 dataset (labels + protected attr only)
train_df = pd.concat(
    [ds.y.rename("Attrition"), ds.A.rename("Gender")],
    axis=1
)

aif = to_aif360_dataset(
    train_df["Attrition"],
    train_df["Gender"],
    label_name="Attrition",
    protected_name="Gender",
)

reweighed = apply_reweighing(
    aif,
    privileged_groups=[{"Gender": "Male"}],
    unprivileged_groups=[{"Gender": "Female"}],
)

sample_weights = extract_weights(reweighed)

pipeline_rw = get_logistic_regression_pipeline(ds.X)
pipeline_rw.fit(ds.X, ds.y, model__sample_weight=sample_weights)

pre = pipeline_rw.named_steps["preprocess"]
model = pipeline_rw.named_steps["model"]

X_enc = pre.transform(ds.X).toarray()

explainer = shap.Explainer(
    model,
    X_enc,
    feature_names=pre.get_feature_names_out()
)

shap_vals_rw = explainer(X_enc)

shap.summary_plot(
    shap_vals_rw,
    X_enc,
    feature_names=pre.get_feature_names_out(),
    show=False
)

plt.tight_layout()
plt.savefig("../runs/shap_mitigated/shap_reweighing_summary.png", dpi=300)
plt.close()



## Exponentiated Gradient (In-processing)

This section evaluates how constraint-based fairness optimisation affects feature importance.

In [4]:
from src.mitigation.exponentiated_gradient import train_expgrad
from src.data.preprocessing import build_preprocessor

pre = build_preprocessor(ds.X)
X_enc = pre.fit_transform(ds.X).toarray()

eg_dp = train_expgrad(
    X_enc,
    ds.y,
    ds.A,
    constraint="dp",
    seed=42,
)

best_estimator = eg_dp.predictors_[0]


explainer = shap.Explainer(
    best_estimator,
    X_enc,
    feature_names=pre.get_feature_names_out()
)

shap_vals_eg = explainer(X_enc)

shap.summary_plot(
    shap_vals_eg,
    X_enc,
    feature_names=pre.get_feature_names_out(),
    show=False
)

plt.tight_layout()
plt.savefig("../runs/shap_mitigated/shap_expgrad_dp_summary.png", dpi=300)
plt.close()

from src.mitigation.threshold_optimizer import apply_threshold_optimizer

pipeline_base = get_logistic_regression_pipeline(ds.X)
pipeline_base.fit(ds.X, ds.y)

pre = pipeline_base.named_steps["preprocess"]
model = pipeline_base.named_steps["model"]

X_enc = pre.transform(ds.X).toarray()

post = apply_threshold_optimizer(
    model,
    X_enc,
    ds.y,
    ds.A,
)

explainer = shap.Explainer(
    model,
    X_enc,
    feature_names=pre.get_feature_names_out()
)

shap_vals_th = explainer(X_enc)

shap.summary_plot(
    shap_vals_th,
    X_enc,
    feature_names=pre.get_feature_names_out(),
    show=False
)

plt.tight_layout()
plt.savefig("../runs/shap_mitigated/shap_threshold_summary.png", dpi=300)
plt.close()


## Key Observation

Comparison with baseline SHAP results shows that mitigation does not substantially alter global feature importance rankings.

Core predictors such as tenure, role, and overtime remain dominant across all models. This suggests that fairness interventions primarily affect decision thresholds rather than fundamentally changing learned feature relationships.

This supports the claim that fairness improvements arise from constrained optimisation rather than redistribution of feature importance.